### Chain of Density 기반 커리큘럼 생성 아이디어
- 초기 커리큘럼 초안 생성: 텍스트 전체를 간단히 요약해 주요 주제를 추출합니다. 이 단계는 entry-level outline 을 생성합니다.

- 엔티티 보강 반복: 각 주제 또는 강의 항목에 대해 누락된 개념(엔티티)을 점진적으로 보완합니다. 각 반복에서 커리큘럼 항목의 깊이를 강화합니다.

- 최종 커리큘럼 완성: 반복적으로 내용을 보강한 후, 전체 커리큘럼을 재정렬하고 정제하여 학습 흐름에 맞게 작성 합니다.

In [4]:
from langchain import hub
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import SimpleJsonOutputParser

from dotenv import load_dotenv
import os


# API 키 로드
load_dotenv('../../.env')
api_key = os.getenv("OPENAI_API_KEY")


In [5]:
# 모델 및 파서 설정
llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0, openai_api_key=api_key)

parser = SimpleJsonOutputParser()

# Chain of Density 기본 프롬프트 로딩
cod_prompt = hub.pull("teddynote/chain-of-density-prompt")

# 템플릿 정의
def build_curriculum_prompt(content: str) -> str:
    return f"""
당신은 기술 교육 전문가이며, 아래 기술 콘텐츠를 기반으로 커리큘럼을 작성해야 합니다.

## 요청사항:

1. 전체 커리큘럼의 목차를 생성하세요. (5~7개의 모듈 또는 장으로 구성)
2. 각 항목에는 다음 내용을 포함해야 합니다:
   - 기술의 탄생 배경 또는 등장 배경
   - 주요 키워드 및 전문 용어
   - 핵심 기능이나 주요 구성요소
   - 확장 가능한 기능, 연계 기술, 또는 진화 방향

전체 결과는 한국어로 작성하며, 먼저는 목차부터 소개해주고, 초급부터 고급 수준까지 점진적으로 학습할 수 있는 구조로 구성해주세요.

추가적으로, 반복적으로 내용을 보완하며 누락된 중요한 개념을 통합해 점진적으로 더 밀도 있는 커리큘럼을 만들어주세요.

## 입력 콘텐츠:
{content}
"""

/home/jarvis/.local/lib/python3.12/site-packages/langsmith/client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [6]:


# 커리큘럼 생성 체인
curriculum_chain = (
    {
        "content": lambda d: build_curriculum_prompt(d.get("content")),
        "content_category": lambda d: "기술 교육",
        "entity_range": lambda d: "3-5",
        "max_words": lambda d: 200,
        "iterations": lambda d: 4,
    }
    | cod_prompt
    | llm
    | parser
)

# 최종 결과 요약만 추출
curriculum_final = curriculum_chain | (
    lambda output: output[-1].get("denser_summary", "요약 없음")
)

# 사용 예시 함수
def generate_curriculum(content: str) -> str:
    return curriculum_final.invoke({"content": content})


In [ ]:
text = """

"""

curriculum = generate_curriculum(text)
print(curriculum)
